# Prepare T-ChIC data for the modality-prediction benchmark

This notebook converts the T-ChIC mouse gastruloid data into the four AnnData
files expected by the OpenProblems `task_predict_modality` benchmark.

The first development task prepared here is:

**H3K4me3 → normalized unspliced RNA**

The H3K4me3 and RNA measurements are paired within cells.

The plate assignments are not created here. They are loaded from
`01_data_split_design.ipynb`, which is the source of truth for the
train/validation/test split.

For development:

- the 70% training partition becomes `train_mod1` / `train_mod2`;
- the 15% validation partition temporarily occupies the benchmark
  `test_mod1` / `test_mod2` slots;
- the final 15% test partition remains untouched.

Both modalities contain:

- `layers["counts"]`: raw counts;
- `layers["normalized"]`: explicit `log_cp10k` values.

The normalization is

$$
x'_{ij}
=
\log\left(
1 +
10000 \cdot
\frac{x_{ij}}{\sum_k x_{ik}}
\right)
$$

which was verified against the OpenProblems benchmark fixture.


## 1. Imports and paths

The notebook is normally opened from the `notebooks/` directory in Jupyter.
The repository root is therefore resolved explicitly rather than relying on
relative output paths.

In [1]:
from pathlib import Path
import gc
import shutil

import anndata as ad
import numpy as np
import pandas as pd
import scipy.sparse as sp

cwd = Path.cwd().resolve()

if cwd.name == "notebooks" and (cwd.parent / "src").is_dir():
    repo_root = cwd.parent
elif (cwd / "src").is_dir() and (cwd / "notebooks").is_dir():
    repo_root = cwd
else:
    raise RuntimeError(
        f"Could not determine repository root from working directory: {cwd}"
    )

RAW_PATH = Path(
    "/home/vivek/NOBINFBACKUP/projects/mGast_tchic/adata_files/"
    "allsamples_genelevel_RNAandChIC_counts.h5ad"
)

FILTERED_PATH = Path(
    "/home/vivek/NOBINFBACKUP/projects/mGast_tchic/adata_files/"
    "allsamples_genelevel_RNAandChIC_counts.filt-norm.h5ad"
)

SPLIT_PATH = (
    repo_root
    / "outputs"
    / "tchic"
    / "splits"
    / "h3k4me3_plate_split.csv"
)

OUT_DIR = (
    repo_root
    / "outputs"
    / "tchic"
    / "prepared"
    / "h3k4me3_genelevel_dev"
)

print("Repository root:", repo_root)
print("Raw source:", RAW_PATH)
print("Filtered source:", FILTERED_PATH)
print("Split:", SPLIT_PATH)
print("Output:", OUT_DIR)

assert RAW_PATH.exists()
assert FILTERED_PATH.exists()
assert SPLIT_PATH.exists()


Repository root: /home/michiel/thesis/benchmark/task_predict_modality_upstream_validation
Raw source: /home/vivek/NOBINFBACKUP/projects/mGast_tchic/adata_files/allsamples_genelevel_RNAandChIC_counts.h5ad
Filtered source: /home/vivek/NOBINFBACKUP/projects/mGast_tchic/adata_files/allsamples_genelevel_RNAandChIC_counts.filt-norm.h5ad
Split: /home/michiel/thesis/benchmark/task_predict_modality_upstream_validation/outputs/tchic/splits/h3k4me3_plate_split.csv
Output: /home/michiel/thesis/benchmark/task_predict_modality_upstream_validation/outputs/tchic/prepared/h3k4me3_genelevel_dev


## 2. Inspect the source datasets

The raw H5AD provides the molecular counts.

The filtered H5AD is used only to define the retained cell and gene population
and to provide observation and feature metadata. Its normalized molecular
layers are not used to construct the benchmark files.

In [2]:
raw = ad.read_h5ad(RAW_PATH, backed="r")
filtered = ad.read_h5ad(FILTERED_PATH, backed="r")

print("Raw shape:", raw.shape)
print("Raw layers:", list(raw.layers.keys()))
print()
print("Filtered shape:", filtered.shape)
print("Filtered layers:", list(filtered.layers.keys()))
print()
print("Raw cells:", raw.n_obs)
print("Filtered retained cells:", filtered.n_obs)
print("Raw genes:", raw.n_vars)
print("Filtered retained genes:", filtered.n_vars)

assert "chic_counts" in raw.layers
assert "unspliced" in raw.layers


Raw shape: (46080, 26910)
Raw layers: ['ambiguous', 'chic_counts', 'spliced', 'total', 'unspliced']

Filtered shape: (37265, 24877)
Filtered layers: ['chic', 'spliced', 'total', 'unspliced']

Raw cells: 46080
Filtered retained cells: 37265
Raw genes: 26910
Filtered retained genes: 24877


## 3. Load the canonical plate split

The split was generated by `01_data_split_design.ipynb`.

Only the training and validation partitions are prepared here. The final test
partition is identified and checked, but is not written to the development
benchmark files.

In [3]:
split = pd.read_csv(SPLIT_PATH)

assert set(split["mark"].astype(str)) == {"H3K4me3"}
assert split["sample"].is_unique

print(split["split"].value_counts())
display(split.head())

train_samples = set(
    split.loc[split["split"] == "train", "sample"].astype(str)
)

validation_samples = set(
    split.loc[split["split"] == "validation", "sample"].astype(str)
)

final_test_samples = set(
    split.loc[split["split"] == "test", "sample"].astype(str)
)

assert len(train_samples) == 42
assert len(validation_samples) == 9
assert len(final_test_samples) == 9


split
train         42
test           9
validation     9
Name: count, dtype: int64


,sample,n_cells,plate_batch,day,rep,mark,plate_grp,seqrun,exp_batch,day_rep,batch_day_rep,split
0,Gastd3-rep3-H3K4me3-1,304,6,3,rep3,H3K4me3,1,OUDxxxx,2022,3_rep3,3_rep3_rep3,train
1,Gastd3-rep3-H3K4me3-2,321,7,3,rep3,H3K4me3,2,OUDxxxx,2022,3_rep3,3_rep3_rep3,test
2,Gastd3-rep3-H3K4me3-3,314,8,3,rep3,H3K4me3,3,OUDxxxx,2022,3_rep3,3_rep3_rep3,train
3,Gastd3-rep3-H3K4me3-4,332,9,3,rep3,H3K4me3,4,OUDxxxx,2022,3_rep3,3_rep3_rep3,validation
4,Gastd3-rep3-H3K4me3-5,338,10,3,rep3,H3K4me3,5,OUDxxxx,2022,3_rep3,3_rep3_rep3,train


## 4. Determine the retained cells

Cell membership is determined using the retained-cell population of the
filtered dataset.

All cells belonging to one plate receive the same split because the split was
defined at plate level.

In [4]:
filtered_samples = filtered.obs["sample"].astype(str)
filtered_marks = filtered.obs["mark"].astype(str)

train_mask = (
    (filtered_marks == "H3K4me3")
    & filtered_samples.isin(train_samples)
)

validation_mask = (
    (filtered_marks == "H3K4me3")
    & filtered_samples.isin(validation_samples)
)

final_test_mask = (
    (filtered_marks == "H3K4me3")
    & filtered_samples.isin(final_test_samples)
)

train_cells = list(filtered.obs_names[train_mask].astype(str))
validation_cells = list(filtered.obs_names[validation_mask].astype(str))
final_test_cells = list(filtered.obs_names[final_test_mask].astype(str))

print("Training cells:", len(train_cells))
print("Validation cells:", len(validation_cells))
print("Untouched final-test cells:", len(final_test_cells))

assert len(train_cells) == 12251
assert len(validation_cells) == 2632
assert len(final_test_cells) == 2615

assert set(train_cells).isdisjoint(validation_cells)
assert set(train_cells).isdisjoint(final_test_cells)
assert set(validation_cells).isdisjoint(final_test_cells)


Training cells: 12251
Validation cells: 2632
Untouched final-test cells: 2615


## 5. Align retained cells and genes to the raw-count file

The raw and filtered H5AD files do not have to use the same physical ordering.

Cell and gene identifiers are therefore aligned explicitly rather than relying
on row or column position.

In [5]:
genes = list(filtered.var_names.astype(str))

raw_cell_lookup = {
    str(name): i
    for i, name in enumerate(raw.obs_names)
}

raw_gene_lookup = {
    str(name): i
    for i, name in enumerate(raw.var_names)
}

missing_cells = [
    cell
    for cell in filtered.obs_names.astype(str)
    if cell not in raw_cell_lookup
]

missing_genes = [
    gene
    for gene in genes
    if gene not in raw_gene_lookup
]

print("Missing retained cells in raw file:", len(missing_cells))
print("Missing retained genes in raw file:", len(missing_genes))

assert not missing_cells
assert not missing_genes

raw_gene_idx = np.array(
    [raw_gene_lookup[gene] for gene in genes],
    dtype=int,
)

print("Retained genes:", len(genes))


Missing retained cells in raw file: 0
Missing retained genes in raw file: 0
Retained genes: 24877


## 6. Define the benchmark normalization

OpenProblems uses `log_cp10k` for the modality-prediction benchmark.

For every cell:

1. sum the raw counts across features;
2. divide each feature by that cell total;
3. multiply by 10,000;
4. apply `log(1 + x)`.

Zero-count cells remain zero.

This implementation operates directly on sparse matrices.

In [6]:
def materialize_counts(x):
    if hasattr(x, "to_memory"):
        x = x.to_memory()

    if not sp.issparse(x):
        x = sp.csr_matrix(x)
    else:
        x = x.tocsr()

    x = x.astype(np.float64)

    if x.nnz:
        np.testing.assert_allclose(
            x.data,
            np.rint(x.data),
            rtol=0,
            atol=1e-6,
        )

    return x.astype(np.int32)


def log_cp10k(counts):
    counts_float = counts.astype(np.float64)

    row_sums = np.asarray(
        counts_float.sum(axis=1)
    ).ravel()

    scale = np.zeros(
        counts_float.shape[0],
        dtype=np.float64,
    )

    nonzero = row_sums > 0
    scale[nonzero] = 1e4 / row_sums[nonzero]

    normalized = (
        counts_float
        .multiply(scale[:, None])
        .tocsr()
    )

    normalized.data = np.log1p(normalized.data)

    return normalized


# Small mathematical sanity check.
example = sp.csr_matrix(
    np.array([
        [10, 0, 30],
        [0, 0, 0],
    ], dtype=np.int32)
)

example_norm = log_cp10k(example)

expected_first_row = np.log1p(
    np.array([10, 0, 30]) / 40 * 10000
)

np.testing.assert_allclose(
    example_norm.toarray()[0],
    expected_first_row,
)

np.testing.assert_array_equal(
    example_norm.toarray()[1],
    np.zeros(3),
)

print("log_cp10k implementation sanity check passed.")


log_cp10k implementation sanity check passed.


## 7. Construct one benchmark AnnData object

The inherited OpenProblems schema requires `hvg` and `hvg_score` fields.

No HVG selection is performed at this stage, so these are explicit compatibility
placeholders:

- `hvg = False`
- `hvg_score = 0.0`

Methods used in the initial benchmark such as mean-per-gene and SimpleMLP do
not use these fields.

In [7]:
def make_benchmark_adata(
    cells,
    raw_layer,
    modality,
):
    raw_cell_idx = np.array(
        [raw_cell_lookup[cell] for cell in cells],
        dtype=int,
    )

    filtered_cell_idx = filtered.obs_names.get_indexer(cells)

    assert np.all(filtered_cell_idx >= 0)

    counts = materialize_counts(
        raw[
            raw_cell_idx,
            raw_gene_idx,
        ].layers[raw_layer]
    )

    normalized = log_cp10k(counts)

    assert counts.shape == normalized.shape
    assert counts.shape == (len(cells), len(genes))

    obs = filtered.obs.iloc[
        filtered_cell_idx
    ].copy()

    obs["batch"] = obs["sample"].astype(str)

    var = filtered.var.copy()

    var["gene_ids"] = var.index.astype(str)

    var["hvg"] = False
    var["hvg_score"] = 0.0

    uns = {
        "dataset_id":
            "tchic_mgast_h3k4me3_genelevel_dev",
        "common_dataset_id":
            "tchic_mgast_genelevel",
        "modality":
            modality,
        "normalization_id":
            "log_cp10k",
        "dataset_name":
            "T-ChIC mouse gastruloid H3K4me3 development split",
        "dataset_summary":
            "Gene-level H3K4me3 and unspliced RNA from paired single-cell T-ChIC.",
        "dataset_description":
            "Development benchmark using the plate-aware training and "
            "validation partitions of the H3K4me3 T-ChIC gastruloid dataset. "
            "The final test partition is excluded.",
        "dataset_organism":
            "Mus musculus",
    }

    out = ad.AnnData(
        X=None,
        obs=obs,
        var=var,
        layers={
            "counts": counts,
            "normalized": normalized,
        },
        uns=uns,
    )

    assert list(out.obs_names.astype(str)) == cells
    assert list(out.var_names.astype(str)) == genes
    assert np.issubdtype(
        out.layers["counts"].dtype,
        np.integer,
    )
    assert np.issubdtype(
        out.layers["normalized"].dtype,
        np.floating,
    )

    return out


## 8. Build the four development benchmark files

The validation cells occupy the OpenProblems `test_*` slots so that the
benchmark infrastructure can score development results.

They must therefore be described as **validation/development results**, not as
final test results.

In [8]:
print("Creating train_mod1: H3K4me3")
train_mod1 = make_benchmark_adata(
    train_cells,
    raw_layer="chic_counts",
    modality="H3K4me3",
)

print("Creating train_mod2: unspliced RNA")
train_mod2 = make_benchmark_adata(
    train_cells,
    raw_layer="unspliced",
    modality="GEX",
)

print("Creating test_mod1: H3K4me3 validation input")
test_mod1 = make_benchmark_adata(
    validation_cells,
    raw_layer="chic_counts",
    modality="H3K4me3",
)

print("Creating test_mod2: unspliced RNA validation truth")
test_mod2 = make_benchmark_adata(
    validation_cells,
    raw_layer="unspliced",
    modality="GEX",
)

print()
print("Shapes:")
print("train_mod1:", train_mod1.shape)
print("train_mod2:", train_mod2.shape)
print("test_mod1:", test_mod1.shape)
print("test_mod2:", test_mod2.shape)


Creating train_mod1: H3K4me3
Creating train_mod2: unspliced RNA
Creating test_mod1: H3K4me3 validation input
Creating test_mod2: unspliced RNA validation truth

Shapes:
train_mod1: (12251, 24877)
train_mod2: (12251, 24877)
test_mod1: (2632, 24877)
test_mod2: (2632, 24877)


## 9. Validate the benchmark contract

Before writing any files, verify:

- paired files have exactly the same cells and cell order;
- feature order is stable between train and validation;
- training and validation cells do not overlap;
- final-test cells occur in none of the prepared objects;
- raw counts are integer-valued;
- normalized values are floating-point;
- both modalities are labelled `log_cp10k`.


In [9]:
assert list(train_mod1.obs_names) == list(train_mod2.obs_names)
assert list(test_mod1.obs_names) == list(test_mod2.obs_names)

assert list(train_mod1.var_names) == list(test_mod1.var_names)
assert list(train_mod2.var_names) == list(test_mod2.var_names)

train_obs = set(train_mod1.obs_names.astype(str))
validation_obs = set(test_mod1.obs_names.astype(str))
final_test_obs = set(final_test_cells)

assert train_obs.isdisjoint(validation_obs)
assert train_obs.isdisjoint(final_test_obs)
assert validation_obs.isdisjoint(final_test_obs)

for name, obj in {
    "train_mod1": train_mod1,
    "train_mod2": train_mod2,
    "test_mod1": test_mod1,
    "test_mod2": test_mod2,
}.items():

    assert obj.X is None

    assert "counts" in obj.layers
    assert "normalized" in obj.layers

    assert np.issubdtype(
        obj.layers["counts"].dtype,
        np.integer,
    )

    assert np.issubdtype(
        obj.layers["normalized"].dtype,
        np.floating,
    )

    assert obj.uns["normalization_id"] == "log_cp10k"

    assert "batch" in obj.obs
    assert "gene_ids" in obj.var
    assert "hvg" in obj.var
    assert "hvg_score" in obj.var

print("ALL PRE-WRITE BENCHMARK CHECKS PASSED")
print()
print("Training cells:", len(train_obs))
print("Validation cells:", len(validation_obs))
print("Untouched final-test cells:", len(final_test_obs))
print("Prepared/final-test overlap:", len((train_obs | validation_obs) & final_test_obs))


ALL PRE-WRITE BENCHMARK CHECKS PASSED

Training cells: 12251
Validation cells: 2632
Untouched final-test cells: 2615
Prepared/final-test overlap: 0


## 10. Write the prepared H5AD files

The generated files are reproducible outputs of this notebook and should not be
edited manually.

In [10]:
OUT_DIR.mkdir(parents=True, exist_ok=True)

free_gb = shutil.disk_usage(OUT_DIR).free / 1024**3

print(f"Free disk space: {free_gb:.1f} GB")

if free_gb < 15:
    raise RuntimeError(
        "Less than 15 GB free. Stop before writing benchmark files."
    )

objects = {
    "train_mod1.h5ad": train_mod1,
    "train_mod2.h5ad": train_mod2,
    "test_mod1.h5ad": test_mod1,
    "test_mod2.h5ad": test_mod2,
}

for filename, obj in objects.items():
    output_path = OUT_DIR / filename

    print("Writing", output_path.relative_to(repo_root))

    obj.write_h5ad(
        output_path,
        compression="lzf",
    )

print("Finished writing prepared benchmark files.")


Free disk space: 10534.0 GB
Writing outputs/tchic/prepared/h3k4me3_genelevel_dev/train_mod1.h5ad
Writing outputs/tchic/prepared/h3k4me3_genelevel_dev/train_mod2.h5ad
Writing outputs/tchic/prepared/h3k4me3_genelevel_dev/test_mod1.h5ad
Writing outputs/tchic/prepared/h3k4me3_genelevel_dev/test_mod2.h5ad
Finished writing prepared benchmark files.


## 11. Validate the files after writing

The files are opened again from disk to verify that the saved representation
has the expected shapes, metadata, modalities and normalization identifiers.

In [11]:
rows = []

for filename in [
    "train_mod1.h5ad",
    "train_mod2.h5ad",
    "test_mod1.h5ad",
    "test_mod2.h5ad",
]:
    path = OUT_DIR / filename
    obj = ad.read_h5ad(path, backed="r")

    rows.append({
        "file": filename,
        "n_cells": obj.n_obs,
        "n_features": obj.n_vars,
        "modality": obj.uns["modality"],
        "normalization_id": obj.uns["normalization_id"],
        "size_gb": path.stat().st_size / 1024**3,
    })

    assert obj.uns["normalization_id"] == "log_cp10k"
    assert set(obj.layers.keys()) == {"counts", "normalized"}

    obj.file.close()

summary = pd.DataFrame(rows)

display(summary)

summary_path = OUT_DIR / "preparation_summary.csv"
summary.to_csv(summary_path, index=False)

print("Summary:", summary_path.relative_to(repo_root))
print()
print("Final-test cells written: 0")
print("POST-WRITE VALIDATION PASSED")


,file,n_cells,n_features,modality,normalization_id,size_gb
0,train_mod1.h5ad,12251,24877,H3K4me3,log_cp10k,0.110919
1,train_mod2.h5ad,12251,24877,GEX,log_cp10k,0.373693
2,test_mod1.h5ad,2632,24877,H3K4me3,log_cp10k,0.027360
3,test_mod2.h5ad,2632,24877,GEX,log_cp10k,0.085528


Summary: outputs/tchic/prepared/h3k4me3_genelevel_dev/preparation_summary.csv

Final-test cells written: 0
POST-WRITE VALIDATION PASSED


## 12. Development data contract

The resulting development benchmark has:

- **training:** 42 plates / 12,251 cells;
- **validation:** 9 plates / 2,632 cells;
- **untouched final test:** 9 plates / 2,615 cells.

Both H3K4me3 and unspliced RNA are represented using raw counts plus an
explicit `log_cp10k` normalized layer.

The final-test cells are not written by this notebook.
